# Working Notebook for Querying Player Metrics and Standardizing Against Conference Baselines


In [76]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import glob

In [77]:
'''file_paths = glob.glob('/Users/longer/uconn/player_data/*.csv')
dataframes = [pd.read_csv(file, index_col='Team') for file in file_paths]

player_df = pd.concat(dataframes, axis=1)
player_df = player_df.loc[:, ~player_df.columns.duplicated()]
player_df = player_df.reset_index()
player_df.to_csv("master_player.csv", index=False)'''

'file_paths = glob.glob(\'/Users/longer/uconn/player_data/*.csv\')\ndataframes = [pd.read_csv(file, index_col=\'Team\') for file in file_paths]\n\nplayer_df = pd.concat(dataframes, axis=1)\nplayer_df = player_df.loc[:, ~player_df.columns.duplicated()]\nplayer_df = player_df.reset_index()\nplayer_df.to_csv("master_player.csv", index=False)'

In [78]:
df = pd.read_csv('../datasets/master_player.csv')

df.head(25)

,Team,Player,Position,TOI (sec),TOI (min),Total Games played,Total TOI/GP (sec),Total TOI/GP (min),Successful OZ Offensive Touches,Failed OZ Offensive Touches,...,Defensive Dump-In Recovery Exit Rate,Total Dump Out Attempts,Dump Out Rate,Successful Dump Out Attempts,Dump Out Success Rate,Total Icings,Successful OZ Body Checks,Successful DZ Body Checks,Successful NZ Body Checks,Successful Body Checks
0,Stonehill College Skyhawks,Zach Aben,F,8113.700000,135:14,8,1014.212500,16:54,54,45,...,0.545455,15,0.288462,11,0.733333,2,2,2,0,4
1,College of the Holy Cross Crusaders,Michael Abgrall,F,32080.833333,534:41,37,867.049550,14:27,343,189,...,0.576923,60,0.281690,44,0.733333,7,4,3,1,8
2,Ohio State University Buckeyes,Chris Able,D,36097.300000,601:37,33,1093.857576,18:14,133,86,...,0.756410,102,0.432203,70,0.686275,10,0,9,0,9
3,Michigan Tech University Huskies,Ryan Abraham,F,4624.800000,77:05,6,770.800000,12:51,38,36,...,1.000000,5,0.178571,4,0.800000,0,1,0,0,1
4,University of Alaska Anchorage Seawolves,Logan Acheson,D,37018.366667,616:58,33,1121.768687,18:42,160,141,...,0.743243,106,0.368056,69,0.650943,12,0,7,0,7
5,Arizona State University Sun Devils,Tony Achille,F,3174.666667,52:55,12,264.555556,4:25,21,12,...,1.000000,3,0.085714,3,1.000000,0,0,0,0,0
6,Providence College Friars,Tanner Adams,F,30581.533333,509:42,35,873.758095,14:34,307,231,...,0.615385,80,0.307692,48,0.600000,2,2,3,1,6
7,Sacred Heart University Pioneers,Mikey Adamson,D,49634.800000,827:15,40,1240.870000,20:41,697,245,...,0.712329,73,0.201657,39,0.534247,20,2,12,1,15
8,UMass Lowell River Hawks,David Adaszynski,F,17938.600000,298:59,28,640.664286,10:41,129,106,...,0.750000,29,0.235772,17,0.586207,3,1,2,1,4
9,University of Vermont Catamounts,Jonah Aegerter,F,27691.866667,461:32,32,865.370833,14:25,297,181,...,0.655172,35,0.159817,24,0.685714,2,2,7,1,10


In [79]:
df.dtypes


Team                          object
Player                        object
Position                      object
TOI (sec)                    float64
TOI (min)                     object
                              ...   
Total Icings                   int64
Successful OZ Body Checks      int64
Successful DZ Body Checks      int64
Successful NZ Body Checks      int64
Successful Body Checks         int64
Length: 394, dtype: object

In [80]:
len(df)

1548

In [81]:
team_df = pd.read_csv('../datasets/ss_cols_df.csv')
team_df = team_df.iloc[:, 1:]

team_df.columns

Index(['Team', 'Conference', 'Successful OZ Open Ice Dekes',
       'Failed OZ Open Ice Dekes', 'Successful Open Ice Dekes',
       'Failed Open Ice Dekes', 'Dump In Rate', 'Total Dump In Attempts',
       'Dump Ins Below Hash Marks', 'Total Chip In Attempts',
       ...
       'Successful NZ Blocked Passes', 'Successful Blocked Passes',
       'DZ Reception Preventions', 'Total Reception Preventions',
       'Total OZ Obstruction Penalties', 'Total OZ Penalties',
       'Total DZ Obstruction Penalties', 'Total NZ Undisciplined Penalties',
       'Total Obstruction Penalties', 'Unnamed: 0'],
      dtype='object', length=179)

In [82]:
common_cols = team_df.columns.intersection(df.columns)


In [83]:
from sklearn.preprocessing import StandardScaler

id_cols = ['Team', 'Player', 'Position', 'TOI (min)']

common_metrics = list(team_df.columns.intersection(df.columns))

cols_to_keep = id_cols + [col for col in common_metrics if col not in id_cols]

player_df = df[cols_to_keep].copy()

player_df.set_index(id_cols, inplace=True)

scaler = StandardScaler()
scaled_data = scaler.fit_transform(player_df)

player_scaled = pd.DataFrame(
    scaled_data, 
    index=player_df.index, 
    columns=player_df.columns
)

player_scaled.head(25)

,,,,Successful OZ Open Ice Dekes,Failed OZ Open Ice Dekes,Successful Open Ice Dekes,Failed Open Ice Dekes,Dump In Rate,Total Dump In Attempts,Dump Ins Below Hash Marks,Total Chip In Attempts,Chip Ins Below Hash Marks,Total Self Chip-In Recoveries,...,Successful OZ Blocked Passes,Successful NZ Blocked Passes,Successful Blocked Passes,DZ Reception Preventions,Total Reception Preventions,Total OZ Obstruction Penalties,Total OZ Penalties,Total DZ Obstruction Penalties,Total NZ Undisciplined Penalties,Total Obstruction Penalties
Team,Player,Position,TOI (min),,,,,,,,,,,,,,,,,,,,,
Stonehill College Skyhawks,Zach Aben,F,135:14,-0.827880,-0.616982,-1.049992,-0.892081,-0.070361,-1.149685,-1.324154,-0.363459,-0.611188,-0.880877,...,-1.335234,-0.947191,-1.285237,-0.234978,-0.198771,-0.817347,-0.490466,-0.165214,-0.627221,-0.736026
College of the Holy Cross Crusaders,Michael Abgrall,F,534:41,-0.169489,0.140600,0.117619,0.191423,-0.937595,-0.168079,-0.119940,-0.213623,-0.272502,-0.095168,...,0.198898,-0.361218,0.014721,-0.796315,-0.986729,3.377790,3.370902,1.247780,1.735158,2.784617
Ohio State University Buckeyes,Chris Able,D,601:37,-0.718148,-0.839800,-0.393211,-0.706338,0.724604,-0.812258,-0.630819,-0.962804,-0.949873,-0.880877,...,-0.601519,0.371249,0.779402,0.326359,0.195208,1.699735,0.474876,-0.871711,-0.627221,0.144135
Michigan Tech University Huskies,Ryan Abraham,F,77:05,-0.608416,-0.795236,-0.685113,-1.015910,-0.485125,-1.487111,-1.506611,-0.663132,-0.780530,-0.488023,...,-1.401936,-1.386671,-1.514641,-0.796315,-0.592750,-0.817347,-0.490466,-0.871711,-0.627221,-1.176106
University of Alaska Anchorage Seawolves,Logan Acheson,D,616:58,-0.498684,-0.349600,0.263570,-0.272936,0.218717,-0.444156,-0.302397,-0.962804,-0.949873,-0.880877,...,0.532405,0.224756,0.932338,2.010368,2.559083,-0.817347,-0.973137,-0.871711,-0.627221,-1.176106
Arizona State University Sun Devils,Tony Achille,F,52:55,-0.827880,-0.928927,-0.977016,-1.201654,0.672983,-1.241710,-1.141698,-0.513296,-0.441845,-0.880877,...,-1.335234,-1.386671,-1.412684,-0.234978,-0.592750,-0.817347,-0.973137,-0.165214,0.553969,-0.736026
Providence College Friars,Tanner Adams,F,509:42,0.159707,0.318854,0.044643,0.315252,-0.513386,0.629475,0.536903,1.584412,1.082241,1.476249,...,0.132196,0.224756,-0.087237,-0.796315,-0.986729,-0.817347,-0.490466,-0.165214,0.553969,-0.736026
Sacred Heart University Pioneers,Mikey Adamson,D,827:15,0.708366,-0.394164,1.139279,0.067594,0.257694,-0.168079,-0.156432,-0.663132,-0.780530,-0.488023,...,0.999314,1.543196,1.314678,-0.796315,-0.592750,0.021680,-0.490466,3.367271,-0.627221,2.344537
UMass Lowell River Hawks,David Adaszynski,F,298:59,-0.059757,-0.349600,-0.247259,-0.118150,-1.391860,-0.996309,-0.995732,-0.663132,-0.611188,0.690541,...,-0.668220,-0.947191,-0.673492,-0.234978,-0.592750,-0.817347,-0.973137,-0.165214,-0.627221,-0.736026


# Create Conference/Team Dictionary

In [84]:

# define teams in each conference
AHA = [
    'Air Force Academy Falcons', 
    'Army Black Knights',
    'Bentley University Falcons',
    'Canisius College Golden Griffins',
    'College of the Holy Cross Crusaders',
    'Mercyhurst University Lakers',
    'Niagara University Purple Eagles',
    'Robert Morris University Colonials',
    'Rochester Institute of Technology Tigers',
    'Sacred Heart University Pioneers'
]

CCHA = [
    'Augustana University Vikings',
    'Bemidji State University Beavers',
    'Bowling Green State University Falcons',
    'Ferris State University Bulldogs',
    'Lake Superior State University Lakers',
    'Michigan Tech University Huskies',
    'Minnesota State University Mavericks',
    'Northern Michigan University Wildcats',
    'The University of St. Thomas Tommies'
]

ECAC = [
    'Brown University Bears',
    'Clarkson University Golden Knights',
    'Colgate University Raiders',
    'Cornell University Big Red',
    'Dartmouth College Big Green',
    'Harvard University Crimson',
    'Princeton University Tigers',
    'Quinnipiac University Bobcats',
    'Rensselaer Polytechnic Institute Engineers',
    'St. Lawrence University Saints',
    'Union College Garnet Chargers',
    'Yale University Bulldogs'
]

HE = [
    'Boston College Eagles',
    'Boston University Terriers',
    'Merrimack College Warriors',
    'Northeastern University Huskies',
    'UMass Lowell River Hawks',
    'UMass Minutemen',
    'University of Connecticut Huskies',
    'University of Maine Black Bears',
    'University of New Hampshire Wildcats',
    'University of Vermont Catamounts',
    'Providence College Friars'
]

NCHC = [
    'Arizona State University Sun Devils',
    'Colorado College Tigers',
    'University of Denver Pioneers',
    'Miami University (Ohio) RedHawks',
    'University of Minnesota-Duluth Bulldogs',
    'University of North Dakota Fighting Hawks',
    'University of Nebraska Omaha Mavericks',
    'St. Cloud State University Huskies',
    'Western Michigan University Broncos'
]

BT = [
    'Michigan State University Spartans',
    'Michigan Wolverines',
    'Ohio State University Buckeyes',
    'Penn State University Nittany Lions',
    'University Of Notre Dame Fighting Irish',
    'University of Minnesota Golden Gophers',
    'University of Wisconsin Badgers'
]

INDE = [
    'University of Alaska Anchorage Seawolves',
    'University of Alaska Fairbanks Nanooks',
    'Lindenwood University Lions',
    'Long Island University Sharks',
    'Stonehill College Skyhawks'
]

In [85]:
conference_mapping = {}
lists = [AHA, CCHA, ECAC, HE, NCHC, BT, INDE]
names = ['Atlantic Hockey', 'CCHA', 'ECAC', 'Hockey East', 'NCHC', 'Big Ten', 'Independent']
for conf_list, conf_name in zip(lists, names):
    for team in conf_list:
        conference_mapping[team] = conf_name

player_scaled = player_scaled.reset_index()
player_scaled['Conference'] = player_scaled['Team'].map(conference_mapping)

player_scaled.head(25)

,Team,Player,Position,TOI (min),Successful OZ Open Ice Dekes,Failed OZ Open Ice Dekes,Successful Open Ice Dekes,Failed Open Ice Dekes,Dump In Rate,Total Dump In Attempts,...,Successful NZ Blocked Passes,Successful Blocked Passes,DZ Reception Preventions,Total Reception Preventions,Total OZ Obstruction Penalties,Total OZ Penalties,Total DZ Obstruction Penalties,Total NZ Undisciplined Penalties,Total Obstruction Penalties,Conference
0,Stonehill College Skyhawks,Zach Aben,F,135:14,-0.827880,-0.616982,-1.049992,-0.892081,-0.070361,-1.149685,...,-0.947191,-1.285237,-0.234978,-0.198771,-0.817347,-0.490466,-0.165214,-0.627221,-0.736026,Independent
1,College of the Holy Cross Crusaders,Michael Abgrall,F,534:41,-0.169489,0.140600,0.117619,0.191423,-0.937595,-0.168079,...,-0.361218,0.014721,-0.796315,-0.986729,3.377790,3.370902,1.247780,1.735158,2.784617,Atlantic Hockey
2,Ohio State University Buckeyes,Chris Able,D,601:37,-0.718148,-0.839800,-0.393211,-0.706338,0.724604,-0.812258,...,0.371249,0.779402,0.326359,0.195208,1.699735,0.474876,-0.871711,-0.627221,0.144135,Big Ten
3,Michigan Tech University Huskies,Ryan Abraham,F,77:05,-0.608416,-0.795236,-0.685113,-1.015910,-0.485125,-1.487111,...,-1.386671,-1.514641,-0.796315,-0.592750,-0.817347,-0.490466,-0.871711,-0.627221,-1.176106,CCHA
4,University of Alaska Anchorage Seawolves,Logan Acheson,D,616:58,-0.498684,-0.349600,0.263570,-0.272936,0.218717,-0.444156,...,0.224756,0.932338,2.010368,2.559083,-0.817347,-0.973137,-0.871711,-0.627221,-1.176106,Independent
5,Arizona State University Sun Devils,Tony Achille,F,52:55,-0.827880,-0.928927,-0.977016,-1.201654,0.672983,-1.241710,...,-1.386671,-1.412684,-0.234978,-0.592750,-0.817347,-0.973137,-0.165214,0.553969,-0.736026,NCHC
6,Providence College Friars,Tanner Adams,F,509:42,0.159707,0.318854,0.044643,0.315252,-0.513386,0.629475,...,0.224756,-0.087237,-0.796315,-0.986729,-0.817347,-0.490466,-0.165214,0.553969,-0.736026,Hockey East
7,Sacred Heart University Pioneers,Mikey Adamson,D,827:15,0.708366,-0.394164,1.139279,0.067594,0.257694,-0.168079,...,1.543196,1.314678,-0.796315,-0.592750,0.021680,-0.490466,3.367271,-0.627221,2.344537,Atlantic Hockey
8,UMass Lowell River Hawks,David Adaszynski,F,298:59,-0.059757,-0.349600,-0.247259,-0.118150,-1.391860,-0.996309,...,-0.947191,-0.673492,-0.234978,-0.592750,-0.817347,-0.973137,-0.165214,-0.627221,-0.736026,Hockey East
9,University of Vermont Catamounts,Jonah Aegerter,F,461:32,2.134880,0.809054,1.869035,0.934398,-1.147833,0.015972,...,-0.361218,-0.291152,-0.796315,-0.986729,0.021680,-0.490466,-0.871711,-0.627221,0.144135,Hockey East


In [88]:
conference_col = player_scaled.pop('Conference')
player_scaled.insert(1, 'Conference', conference_col)
player_scaled.head(25)

,Team,Conference,Player,Position,TOI (min),Successful OZ Open Ice Dekes,Failed OZ Open Ice Dekes,Successful Open Ice Dekes,Failed Open Ice Dekes,Dump In Rate,...,Successful OZ Blocked Passes,Successful NZ Blocked Passes,Successful Blocked Passes,DZ Reception Preventions,Total Reception Preventions,Total OZ Obstruction Penalties,Total OZ Penalties,Total DZ Obstruction Penalties,Total NZ Undisciplined Penalties,Total Obstruction Penalties
0,Stonehill College Skyhawks,Independent,Zach Aben,F,135:14,-0.827880,-0.616982,-1.049992,-0.892081,-0.070361,...,-1.335234,-0.947191,-1.285237,-0.234978,-0.198771,-0.817347,-0.490466,-0.165214,-0.627221,-0.736026
1,College of the Holy Cross Crusaders,Atlantic Hockey,Michael Abgrall,F,534:41,-0.169489,0.140600,0.117619,0.191423,-0.937595,...,0.198898,-0.361218,0.014721,-0.796315,-0.986729,3.377790,3.370902,1.247780,1.735158,2.784617
2,Ohio State University Buckeyes,Big Ten,Chris Able,D,601:37,-0.718148,-0.839800,-0.393211,-0.706338,0.724604,...,-0.601519,0.371249,0.779402,0.326359,0.195208,1.699735,0.474876,-0.871711,-0.627221,0.144135
3,Michigan Tech University Huskies,CCHA,Ryan Abraham,F,77:05,-0.608416,-0.795236,-0.685113,-1.015910,-0.485125,...,-1.401936,-1.386671,-1.514641,-0.796315,-0.592750,-0.817347,-0.490466,-0.871711,-0.627221,-1.176106
4,University of Alaska Anchorage Seawolves,Independent,Logan Acheson,D,616:58,-0.498684,-0.349600,0.263570,-0.272936,0.218717,...,0.532405,0.224756,0.932338,2.010368,2.559083,-0.817347,-0.973137,-0.871711,-0.627221,-1.176106
5,Arizona State University Sun Devils,NCHC,Tony Achille,F,52:55,-0.827880,-0.928927,-0.977016,-1.201654,0.672983,...,-1.335234,-1.386671,-1.412684,-0.234978,-0.592750,-0.817347,-0.973137,-0.165214,0.553969,-0.736026
6,Providence College Friars,Hockey East,Tanner Adams,F,509:42,0.159707,0.318854,0.044643,0.315252,-0.513386,...,0.132196,0.224756,-0.087237,-0.796315,-0.986729,-0.817347,-0.490466,-0.165214,0.553969,-0.736026
7,Sacred Heart University Pioneers,Atlantic Hockey,Mikey Adamson,D,827:15,0.708366,-0.394164,1.139279,0.067594,0.257694,...,0.999314,1.543196,1.314678,-0.796315,-0.592750,0.021680,-0.490466,3.367271,-0.627221,2.344537
8,UMass Lowell River Hawks,Hockey East,David Adaszynski,F,298:59,-0.059757,-0.349600,-0.247259,-0.118150,-1.391860,...,-0.668220,-0.947191,-0.673492,-0.234978,-0.592750,-0.817347,-0.973137,-0.165214,-0.627221,-0.736026
9,University of Vermont Catamounts,Hockey East,Jonah Aegerter,F,461:32,2.134880,0.809054,1.869035,0.934398,-1.147833,...,0.132196,-0.361218,-0.291152,-0.796315,-0.986729,0.021680,-0.490466,-0.871711,-0.627221,0.144135


# Data Cleaning which includes alignment, column formatting, and dtype standardizing

In [89]:
player_scaled['Conference'].value_counts()

Conference
ECAC               294
Hockey East        266
Atlantic Hockey    257
CCHA               226
NCHC               217
Big Ten            160
Independent        128
Name: count, dtype: int64

In [91]:
conference_baseline = pd.read_csv('../datasets/conference_baselines.csv')

conference_baseline


,Conference,Successful OZ Open Ice Dekes,Failed OZ Open Ice Dekes,OZ Open Ice Deke Success Rate,Successful DZ Open Ice Dekes,Failed DZ Open Ice Dekes,DZ Open Ice Deke Success Rate,Successful NZ Open Ice Dekes,Failed NZ Open Ice Dekes,NZ Open Ice Deke Success Rate,...,Total DZ Undisciplined Penalties,Total DZ Obstruction Penalties,Total DZ Penalties,Total NZ Undisciplined Penalties,Total NZ Obstruction Penalties,Total NZ Penalties,Total Undisciplined Penalties,Total Obstruction Penalties,Total Penalties,Unnamed: 0
0,Atlantic Hockey,0.012793,0.044063,-0.048808,0.382111,0.522200,0.120763,0.373259,0.307125,0.179055,...,0.292546,0.714577,0.663367,-0.093930,0.347758,0.170320,0.224544,0.837592,0.618701,-0.615918
1,Big Ten,1.005668,0.656809,0.767877,0.302785,-0.150096,0.530402,0.664389,0.530538,0.451398,...,0.017465,-0.582389,-0.332652,0.527552,-0.327604,0.134070,0.245668,-0.187644,0.093687,0.472937
2,CCHA,-0.193569,-0.239019,-0.013960,0.160725,0.401156,-0.044203,-0.253110,-0.278863,-0.122224,...,-0.043664,0.498416,0.261346,0.167273,0.338753,0.339490,-0.039119,-0.048157,-0.056142,-0.604919
3,ECAC,-0.454442,-0.198017,-0.467512,-0.347373,0.004029,-0.507456,-0.269652,-0.007119,-0.337996,...,-0.120075,-0.027765,-0.114137,-0.598321,-0.314097,-0.612089,-0.500725,-0.309696,-0.556594,-0.151230
4,Hockey East,0.202873,-0.020888,0.302585,0.020267,-0.249381,0.224372,0.013558,-0.278179,0.215398,...,-0.353097,-0.241082,-0.430590,-0.283077,-0.455309,-0.495373,-0.244673,-0.216176,-0.306229,0.354953
5,Independent,-0.867945,-1.278178,-0.084133,-0.882432,-1.044662,-0.462873,-0.605995,-1.053661,0.111626,...,0.417582,-1.094349,-0.312325,1.176055,-0.327604,0.569077,0.879399,-1.191958,0.060528,0.307959
6,NCHC,0.271071,0.619598,-0.151236,0.103694,-0.013503,0.166569,0.121829,0.514741,-0.191079,...,-0.154807,-0.108351,-0.190363,-0.072914,0.368769,0.198515,-0.273834,0.463299,0.029825,0.507155


In [92]:
player_conference_baselines = player_scaled.groupby('Conference').mean(numeric_only=True)
player_conference_baselines

,Successful OZ Open Ice Dekes,Failed OZ Open Ice Dekes,Successful Open Ice Dekes,Failed Open Ice Dekes,Dump In Rate,Total Dump In Attempts,Dump Ins Below Hash Marks,Total Chip In Attempts,Chip Ins Below Hash Marks,Total Self Chip-In Recoveries,...,Successful OZ Blocked Passes,Successful NZ Blocked Passes,Successful Blocked Passes,DZ Reception Preventions,Total Reception Preventions,Total OZ Obstruction Penalties,Total OZ Penalties,Total DZ Obstruction Penalties,Total NZ Undisciplined Penalties,Total Obstruction Penalties
Conference,,,,,,,,,,,,,,,,,,,,,
Atlantic Hockey,-0.039262,-0.044417,-0.009024,-0.025639,0.066830,0.029459,0.022901,0.055732,0.050370,-0.015680,...,-0.074396,-0.071651,-0.065814,-0.005638,-0.070000,0.047798,0.027889,0.071201,-0.043520,0.082489
Big Ten,0.268067,0.201596,0.261290,0.201678,-0.253540,0.029776,0.029902,-0.102182,-0.103159,-0.104990,...,0.213905,0.205528,0.217520,0.171991,0.222294,0.116071,0.212424,0.011410,0.170082,0.089125
CCHA,-0.056358,-0.058950,-0.049321,-0.047194,0.122474,0.107183,0.099492,0.107928,0.104398,0.115165,...,0.085859,0.064002,0.061075,0.177331,0.169059,-0.171370,-0.125259,0.062990,0.015639,-0.033066
ECAC,-0.084018,-0.025680,-0.088401,-0.015485,0.013885,-0.094835,-0.099709,-0.083663,-0.085303,-0.080470,...,-0.122585,-0.133007,-0.110212,-0.141422,-0.117027,-0.052519,-0.106299,-0.004210,-0.104926,-0.050459
Hockey East,0.054513,0.015453,0.046564,-0.000838,0.009769,-0.014588,0.003115,0.004935,0.011434,0.000830,...,0.036658,0.001161,0.011942,-0.002846,-0.032885,0.050068,0.044827,-0.024446,-0.041067,-0.016346
Independent,-0.196922,-0.225657,-0.230726,-0.266164,0.032033,-0.213852,-0.218866,-0.134023,-0.120358,-0.085961,...,-0.315954,-0.187257,-0.262476,-0.199895,-0.217239,-0.155302,-0.019108,-0.209370,0.175619,-0.240935
NCHC,0.070708,0.114314,0.068184,0.109821,-0.069644,0.104039,0.107584,0.083287,0.080231,0.134751,...,0.148487,0.155897,0.143460,0.008182,0.069934,0.137675,0.041139,0.000829,-0.001245,0.101546


In [93]:
missing_player_cols = player_conference_baselines.columns.difference(conference_baseline)
missing_player_cols

Index(['1 Timers Pass from East-West', 'Base Successful OZ Blocked Passes',
       'Carry Controlled Entries with Play After',
       'Carry Controlled Entries with Scoring Chance After',
       'Carry Controlled Entries with Shot On Net After',
       'Carry-Out with Play After Rate', 'Carry-Outs with Play After',
       'Chip Ins Below Hash Marks', 'Controlled Entries',
       'Controlled Entries with Play After',
       ...
       'Total Offsides', 'Total Outlet Pass Attempts',
       'Total Pass to Slot Attempts', 'Total Reception Preventions',
       'Total Self Chip-In Recoveries', 'Total Shot From Slot Attempts',
       'Total South Pass Attempts in the NZ',
       'Total South Pass Attempts in the OZ',
       'Total Successful Shot From Slot Attempts', 'Total Time of Possession'],
      dtype='object', length=168)

In [94]:
for df in [player_conference_baselines, conference_baseline]:
    df.columns = df.columns.str.strip().str.replace(' ', '_').str.lower()

print(player_conference_baselines.columns)

Index(['successful_oz_open_ice_dekes', 'failed_oz_open_ice_dekes',
       'successful_open_ice_dekes', 'failed_open_ice_dekes', 'dump_in_rate',
       'total_dump_in_attempts', 'dump_ins_below_hash_marks',
       'total_chip_in_attempts', 'chip_ins_below_hash_marks',
       'total_self_chip-in_recoveries',
       ...
       'successful_oz_blocked_passes', 'successful_nz_blocked_passes',
       'successful_blocked_passes', 'dz_reception_preventions',
       'total_reception_preventions', 'total_oz_obstruction_penalties',
       'total_oz_penalties', 'total_dz_obstruction_penalties',
       'total_nz_undisciplined_penalties', 'total_obstruction_penalties'],
      dtype='object', length=168)


In [95]:
common_cols = conference_baseline.columns.intersection(player_conference_baselines.columns)
len(common_cols)

168

# Subtracting Conference Baselines from Player Stat Lines

In [96]:
# last cleanup of conference_baseline df
junk_cols = [col for col in conference_baseline.columns if 'unnamed' in col]
conference_baseline.drop(columns=junk_cols, inplace=True, errors='ignore')

if 'conference' in conference_baseline.columns:
    conference_baseline.set_index('conference', inplace=True)

conference_baseline.index.name = 'conference'
conference_baseline

,successful_oz_open_ice_dekes,failed_oz_open_ice_dekes,oz_open_ice_deke_success_rate,successful_dz_open_ice_dekes,failed_dz_open_ice_dekes,dz_open_ice_deke_success_rate,successful_nz_open_ice_dekes,failed_nz_open_ice_dekes,nz_open_ice_deke_success_rate,successful_open_ice_dekes,...,total_oz_penalties,total_dz_undisciplined_penalties,total_dz_obstruction_penalties,total_dz_penalties,total_nz_undisciplined_penalties,total_nz_obstruction_penalties,total_nz_penalties,total_undisciplined_penalties,total_obstruction_penalties,total_penalties
conference,,,,,,,,,,,,,,,,,,,,,
Atlantic Hockey,0.012793,0.044063,-0.048808,0.382111,0.522200,0.120763,0.373259,0.307125,0.179055,0.216652,...,0.409561,0.292546,0.714577,0.663367,-0.093930,0.347758,0.170320,0.224544,0.837592,0.618701
Big Ten,1.005668,0.656809,0.767877,0.302785,-0.150096,0.530402,0.664389,0.530538,0.451398,0.897750,...,0.464482,0.017465,-0.582389,-0.332652,0.527552,-0.327604,0.134070,0.245668,-0.187644,0.093687
CCHA,-0.193569,-0.239019,-0.013960,0.160725,0.401156,-0.044203,-0.253110,-0.278863,-0.122224,-0.130594,...,-0.599782,-0.043664,0.498416,0.261346,0.167273,0.338753,0.339490,-0.039119,-0.048157,-0.056142
ECAC,-0.454442,-0.198017,-0.467512,-0.347373,0.004029,-0.507456,-0.269652,-0.007119,-0.337996,-0.466706,...,-0.611987,-0.120075,-0.027765,-0.114137,-0.598321,-0.314097,-0.612089,-0.500725,-0.309696,-0.556594
Hockey East,0.202873,-0.020888,0.302585,0.020267,-0.249381,0.224372,0.013558,-0.278179,0.215398,0.140487,...,0.156920,-0.353097,-0.241082,-0.430590,-0.283077,-0.455309,-0.495373,-0.244673,-0.216176,-0.306229
Independent,-0.867945,-1.278178,-0.084133,-0.882432,-1.044662,-0.462873,-0.605995,-1.053661,0.111626,-0.982803,...,0.112982,0.417582,-1.094349,-0.312325,1.176055,-0.327604,0.569077,0.879399,-1.191958,0.060528
NCHC,0.271071,0.619598,-0.151236,0.103694,-0.013503,0.166569,0.121829,0.514741,-0.191079,0.236074,...,0.144715,-0.154807,-0.108351,-0.190363,-0.072914,0.368769,0.198515,-0.273834,0.463299,0.029825


In [97]:
player_deltas = player_scaled.copy()

player_deltas.columns = player_deltas.columns.str.lower().str.strip().str.replace(' ', '_')
player_deltas.set_index('conference', inplace=True)

In [99]:
player_deltas.head(25)

,team,player,position,toi_(min),successful_oz_open_ice_dekes,failed_oz_open_ice_dekes,successful_open_ice_dekes,failed_open_ice_dekes,dump_in_rate,total_dump_in_attempts,...,successful_oz_blocked_passes,successful_nz_blocked_passes,successful_blocked_passes,dz_reception_preventions,total_reception_preventions,total_oz_obstruction_penalties,total_oz_penalties,total_dz_obstruction_penalties,total_nz_undisciplined_penalties,total_obstruction_penalties
conference,,,,,,,,,,,,,,,,,,,,,
Independent,Stonehill College Skyhawks,Zach Aben,F,135:14,-0.827880,-0.616982,-1.049992,-0.892081,-0.070361,-1.149685,...,-1.335234,-0.947191,-1.285237,-0.234978,-0.198771,-0.817347,-0.490466,-0.165214,-0.627221,-0.736026
Atlantic Hockey,College of the Holy Cross Crusaders,Michael Abgrall,F,534:41,-0.169489,0.140600,0.117619,0.191423,-0.937595,-0.168079,...,0.198898,-0.361218,0.014721,-0.796315,-0.986729,3.377790,3.370902,1.247780,1.735158,2.784617
Big Ten,Ohio State University Buckeyes,Chris Able,D,601:37,-0.718148,-0.839800,-0.393211,-0.706338,0.724604,-0.812258,...,-0.601519,0.371249,0.779402,0.326359,0.195208,1.699735,0.474876,-0.871711,-0.627221,0.144135
CCHA,Michigan Tech University Huskies,Ryan Abraham,F,77:05,-0.608416,-0.795236,-0.685113,-1.015910,-0.485125,-1.487111,...,-1.401936,-1.386671,-1.514641,-0.796315,-0.592750,-0.817347,-0.490466,-0.871711,-0.627221,-1.176106
Independent,University of Alaska Anchorage Seawolves,Logan Acheson,D,616:58,-0.498684,-0.349600,0.263570,-0.272936,0.218717,-0.444156,...,0.532405,0.224756,0.932338,2.010368,2.559083,-0.817347,-0.973137,-0.871711,-0.627221,-1.176106
NCHC,Arizona State University Sun Devils,Tony Achille,F,52:55,-0.827880,-0.928927,-0.977016,-1.201654,0.672983,-1.241710,...,-1.335234,-1.386671,-1.412684,-0.234978,-0.592750,-0.817347,-0.973137,-0.165214,0.553969,-0.736026
Hockey East,Providence College Friars,Tanner Adams,F,509:42,0.159707,0.318854,0.044643,0.315252,-0.513386,0.629475,...,0.132196,0.224756,-0.087237,-0.796315,-0.986729,-0.817347,-0.490466,-0.165214,0.553969,-0.736026
Atlantic Hockey,Sacred Heart University Pioneers,Mikey Adamson,D,827:15,0.708366,-0.394164,1.139279,0.067594,0.257694,-0.168079,...,0.999314,1.543196,1.314678,-0.796315,-0.592750,0.021680,-0.490466,3.367271,-0.627221,2.344537
Hockey East,UMass Lowell River Hawks,David Adaszynski,F,298:59,-0.059757,-0.349600,-0.247259,-0.118150,-1.391860,-0.996309,...,-0.668220,-0.947191,-0.673492,-0.234978,-0.592750,-0.817347,-0.973137,-0.165214,-0.627221,-0.736026


In [100]:

player_delta_means = player_deltas.groupby('conference').mean(numeric_only=True)
player_delta_means

,successful_oz_open_ice_dekes,failed_oz_open_ice_dekes,successful_open_ice_dekes,failed_open_ice_dekes,dump_in_rate,total_dump_in_attempts,dump_ins_below_hash_marks,total_chip_in_attempts,chip_ins_below_hash_marks,total_self_chip-in_recoveries,...,successful_oz_blocked_passes,successful_nz_blocked_passes,successful_blocked_passes,dz_reception_preventions,total_reception_preventions,total_oz_obstruction_penalties,total_oz_penalties,total_dz_obstruction_penalties,total_nz_undisciplined_penalties,total_obstruction_penalties
conference,,,,,,,,,,,,,,,,,,,,,
Atlantic Hockey,-0.039262,-0.044417,-0.009024,-0.025639,0.066830,0.029459,0.022901,0.055732,0.050370,-0.015680,...,-0.074396,-0.071651,-0.065814,-0.005638,-0.070000,0.047798,0.027889,0.071201,-0.043520,0.082489
Big Ten,0.268067,0.201596,0.261290,0.201678,-0.253540,0.029776,0.029902,-0.102182,-0.103159,-0.104990,...,0.213905,0.205528,0.217520,0.171991,0.222294,0.116071,0.212424,0.011410,0.170082,0.089125
CCHA,-0.056358,-0.058950,-0.049321,-0.047194,0.122474,0.107183,0.099492,0.107928,0.104398,0.115165,...,0.085859,0.064002,0.061075,0.177331,0.169059,-0.171370,-0.125259,0.062990,0.015639,-0.033066
ECAC,-0.084018,-0.025680,-0.088401,-0.015485,0.013885,-0.094835,-0.099709,-0.083663,-0.085303,-0.080470,...,-0.122585,-0.133007,-0.110212,-0.141422,-0.117027,-0.052519,-0.106299,-0.004210,-0.104926,-0.050459
Hockey East,0.054513,0.015453,0.046564,-0.000838,0.009769,-0.014588,0.003115,0.004935,0.011434,0.000830,...,0.036658,0.001161,0.011942,-0.002846,-0.032885,0.050068,0.044827,-0.024446,-0.041067,-0.016346
Independent,-0.196922,-0.225657,-0.230726,-0.266164,0.032033,-0.213852,-0.218866,-0.134023,-0.120358,-0.085961,...,-0.315954,-0.187257,-0.262476,-0.199895,-0.217239,-0.155302,-0.019108,-0.209370,0.175619,-0.240935
NCHC,0.070708,0.114314,0.068184,0.109821,-0.069644,0.104039,0.107584,0.083287,0.080231,0.134751,...,0.148487,0.155897,0.143460,0.008182,0.069934,0.137675,0.041139,0.000829,-0.001245,0.101546


In [101]:
common_cols = conference_baseline.columns.intersection(player_deltas.columns)

player_deltas.reset_index(inplace=True)

aligned_baselines = player_deltas[['conference']].merge(conference_baseline, on='conference', how='left')

player_deltas[common_cols] = player_deltas[common_cols] - aligned_baselines[common_cols]

player_deltas.head(25)

,conference,team,player,position,toi_(min),successful_oz_open_ice_dekes,failed_oz_open_ice_dekes,successful_open_ice_dekes,failed_open_ice_dekes,dump_in_rate,...,successful_oz_blocked_passes,successful_nz_blocked_passes,successful_blocked_passes,dz_reception_preventions,total_reception_preventions,total_oz_obstruction_penalties,total_oz_penalties,total_dz_obstruction_penalties,total_nz_undisciplined_penalties,total_obstruction_penalties
0,Independent,Stonehill College Skyhawks,Zach Aben,F,135:14,0.040065,0.661197,-0.067189,0.664886,-0.568555,...,-0.081241,-0.314420,0.088066,0.591729,0.658344,-0.147868,-0.603448,0.929134,-1.803276,0.455932
1,Atlantic Hockey,College of the Holy Cross Crusaders,Michael Abgrall,F,534:41,-0.182282,0.096537,-0.099033,-0.094721,-1.400767,...,0.219141,-0.339354,-0.027712,-0.941786,-0.855679,2.927993,2.961341,0.533202,1.829088,1.947025
2,Big Ten,Ohio State University Buckeyes,Chris Able,D,601:37,-1.723816,-1.496609,-1.290960,-1.277219,1.485084,...,-1.144905,-0.321534,0.020158,-0.376204,-0.609392,1.196639,0.010393,-0.289323,-1.154773,0.331779
3,CCHA,Michigan Tech University Huskies,Ryan Abraham,F,77:05,-0.414847,-0.556218,-0.554519,-0.899404,-1.193120,...,-1.990054,-1.865572,-2.185525,-1.758610,-1.508797,0.006107,0.109316,-1.370128,-0.794493,-1.127950
4,Independent,University of Alaska Anchorage Seawolves,Logan Acheson,D,616:58,0.369261,0.928578,1.246374,1.284031,-0.279477,...,1.786398,0.857527,2.305641,2.837076,3.416198,-0.147868,-1.086119,0.222637,-1.803276,0.015851
5,NCHC,Arizona State University Sun Devils,Tony Achille,F,52:55,-1.098951,-1.548525,-1.213090,-1.786195,0.911671,...,-1.888721,-1.987242,-2.152360,-0.202035,-0.826811,-1.474418,-1.117852,-0.056863,0.626882,-1.199325
6,Hockey East,Providence College Friars,Tanner Adams,F,509:42,-0.043166,0.339742,-0.095844,0.486907,-0.308542,...,0.081130,0.336736,0.019922,-0.710410,-0.754288,-1.000650,-0.647386,0.075867,0.837045,-0.519850
7,Atlantic Hockey,Sacred Heart University Pioneers,Mikey Adamson,D,827:15,0.695572,-0.438226,0.922627,-0.218550,-0.205478,...,1.019557,1.565060,1.272246,-0.941786,-0.461700,-0.428117,-0.900027,2.652694,-0.533291,1.506945
8,Hockey East,UMass Lowell River Hawks,David Adaszynski,F,298:59,-0.262630,-0.328712,-0.387747,0.053506,-1.187017,...,-0.719287,-0.835211,-0.566334,-0.149073,-0.360309,-1.000650,-1.130057,0.075867,-0.344144,-0.519850
9,Hockey East,University of Vermont Catamounts,Jonah Aegerter,F,461:32,1.932007,0.829942,1.728548,1.106053,-0.942990,...,0.081130,-0.249237,-0.183993,-0.710410,-0.754288,-0.161623,-0.647386,-0.630630,-0.344144,0.360311


In [102]:
player_deltas.to_csv('ncaa_d1_player_deltas.csv')